In [12]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from typing import Annotated, TypedDict
from langchain_groq import ChatGroq
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv
import time
load_dotenv()

True

In [13]:
class JokeState(TypedDict):
    joke: str
    topic: str
    explanation: str

In [14]:
model = ChatGroq(model='llama-3.3-70b-versatile')

In [15]:
def create_joke(state: JokeState) -> dict:
    joke = model.invoke(f"Tell me a joke about {state['topic']}")
    return {"joke": joke.content}

In [16]:
import random

def explain_joke(state: JokeState) -> dict:
    # Simulate failure with a chance of raising an exception
    # This allows testing fault tolerance without keyboard interrupt
    if random.random() < 0.5:  # 50% chance to fail
        raise Exception("Simulated failure for fault tolerance testing")
    
    time.sleep(2)  # Short delay instead of 60 seconds
    explanation = model.invoke(f"Explain the joke: {state['joke']}")
    return {"explanation": explanation.content}

In [17]:
graph = StateGraph(JokeState)

graph.add_node('create_joke', create_joke)
graph.add_node('explain_joke', explain_joke)

graph.add_edge(START, 'create_joke')
graph.add_edge('create_joke', 'explain_joke')
graph.add_edge('explain_joke', END)

checkpoint = InMemorySaver()                                #Adding persistence to the workflow by creating a checkpoint saver

joke_workflow = graph.compile(checkpointer=checkpoint)

### Create thread for workflow

In [18]:
config1 = {"configurable": {"thread_id": '1'}}

try:
    joke_workflow.invoke({'topic': 'myself'}, config=config1)
except Exception as e:
    print(f"Error during workflow execution: {e}")

Error during workflow execution: Simulated failure for fault tolerance testing


### Get Final State of the workflow thread

In [19]:
joke_workflow.get_state(config1)

StateSnapshot(values={'joke': 'I don\'t know much about you, but I can try to come up with a lighthearted joke. Here\'s one:\n\n"You\'re so unique, you\'re the only person I\'ve met today who\'s talking to a language model. Either that\'s a testament to your curiosity, or you just really love typing to a robot. Either way, I\'m glad you\'re here!"\n\nIf you\'d like, I can try to come up with something more personalized if you give me a bit more information about yourself. What are your interests or hobbies?', 'topic': 'myself'}, next=('explain_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1552e1-eaa6-6ac3-8001-0bc5f3a565d0'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-05-21T15:59:59.653753+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1552e1-e0a5-62db-8000-69a7e3890da7'}}, tasks=(PregelTask(id='aec24e58-3283-9925-d66d-3f99672b9f26', name='explain_joke', path

### Get History of the workflow thread

In [21]:
list(joke_workflow.get_state_history(config1))

[StateSnapshot(values={'joke': 'I don\'t know much about you, but I can try to come up with a lighthearted joke. Here\'s one:\n\n"You\'re so unique, you\'re the only person I\'ve met today who\'s talking to a language model. Either that\'s a testament to your curiosity, or you just really love typing to a robot. Either way, I\'m glad you\'re here!"\n\nIf you\'d like, I can try to come up with something more personalized if you give me a bit more information about yourself. What are your interests or hobbies?', 'topic': 'myself'}, next=('explain_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1552e1-eaa6-6ac3-8001-0bc5f3a565d0'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-05-21T15:59:59.653753+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1552e1-e0a5-62db-8000-69a7e3890da7'}}, tasks=(PregelTask(id='aec24e58-3283-9925-d66d-3f99672b9f26', name='explain_joke', pat

### Re-Run To Check Fault Tolerence

In [22]:
joke_workflow.invoke(None,config=config1)

{'joke': 'I don\'t know much about you, but I can try to come up with a lighthearted joke. Here\'s one:\n\n"You\'re so unique, you\'re the only person I\'ve met today who\'s talking to a language model. Either that\'s a testament to your curiosity, or you just really love typing to a robot. Either way, I\'m glad you\'re here!"\n\nIf you\'d like, I can try to come up with something more personalized if you give me a bit more information about yourself. What are your interests or hobbies?',
 'topic': 'myself',
 'explanation': 'The joke you provided is not a traditional joke with a setup and a punchline, but rather a lighthearted and playful comment. \n\nIt starts by saying "You\'re so unique," which is a common phrase often used to compliment someone. However, instead of saying something like "you\'re so unique because of your personality or style," it takes a humorous turn by saying "you\'re the only person I\'ve met today who\'s talking to a language model." This is a clever and self-a

In [23]:
joke_workflow.get_state(config1)

StateSnapshot(values={'joke': 'I don\'t know much about you, but I can try to come up with a lighthearted joke. Here\'s one:\n\n"You\'re so unique, you\'re the only person I\'ve met today who\'s talking to a language model. Either that\'s a testament to your curiosity, or you just really love typing to a robot. Either way, I\'m glad you\'re here!"\n\nIf you\'d like, I can try to come up with something more personalized if you give me a bit more information about yourself. What are your interests or hobbies?', 'topic': 'myself', 'explanation': 'The joke you provided is not a traditional joke with a setup and a punchline, but rather a lighthearted and playful comment. \n\nIt starts by saying "You\'re so unique," which is a common phrase often used to compliment someone. However, instead of saying something like "you\'re so unique because of your personality or style," it takes a humorous turn by saying "you\'re the only person I\'ve met today who\'s talking to a language model." This is 

In [24]:
list(joke_workflow.get_state_history(config1))

[StateSnapshot(values={'joke': 'I don\'t know much about you, but I can try to come up with a lighthearted joke. Here\'s one:\n\n"You\'re so unique, you\'re the only person I\'ve met today who\'s talking to a language model. Either that\'s a testament to your curiosity, or you just really love typing to a robot. Either way, I\'m glad you\'re here!"\n\nIf you\'d like, I can try to come up with something more personalized if you give me a bit more information about yourself. What are your interests or hobbies?', 'topic': 'myself', 'explanation': 'The joke you provided is not a traditional joke with a setup and a punchline, but rather a lighthearted and playful comment. \n\nIt starts by saying "You\'re so unique," which is a common phrase often used to compliment someone. However, instead of saying something like "you\'re so unique because of your personality or style," it takes a humorous turn by saying "you\'re the only person I\'ve met today who\'s talking to a language model." This is